In [ ]:
import requests, networkx as nx, matplotlib.pyplot as plt, tldextract
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag

start_url, max_pages = 'https://www.manipal.edu/mit.html', 20
domain = lambda u: '.'.join(x for x in tldextract.extract(u)[1:] if x)
main_domain = domain(start_url)
visited, internal_links, to_visit = set(), [], [start_url]
session = requests.Session(); session.headers['User-Agent'] = 'Mozilla/5.0'

while to_visit and len(visited) < max_pages:
    url = urldefrag(to_visit.pop(0))[0]
    if url in visited: continue
    visited.add(url)
    try:
        r = session.get(url, timeout=10); r.raise_for_status()
        if 'text/html' not in r.headers.get('Content-Type', ''): continue
        for a in BeautifulSoup(r.text, 'html.parser').find_all('a', href=True):
            link = urldefrag(urljoin(url, a['href']))[0]
            p = urlparse(link)
            if p.scheme not in ('http', 'https'): continue
            link = p._replace(query='', fragment='').geturl().rstrip('/')
            if domain(link) == main_domain:
                internal_links.append((url.rstrip('/'), link))
                if link not in visited and link not in to_visit: to_visit.append(link)
    except Exception as e:
        print('Skipping', url, e)

print('Pages visited:', len(visited), '| Internal links:', len(internal_links))

In [ ]:
G = nx.DiGraph(internal_links)
print('Graph nodes:', G.number_of_nodes(), '| edges:', G.number_of_edges())

pr = nx.pagerank(G, alpha=0.85)
print('\n--- Top 5 Pages by PageRank ---')
for page, score in sorted(pr.items(), key=lambda x: -x[1])[:5]:
    print(f'{page}: {score:.4f}')

auth = nx.hits(G, max_iter=100)[1]
print('\nTop HITS authority:', max(auth.items(), key=lambda x: x[1]))

In [ ]:
plt.figure(figsize=(12, 8))
nx.draw(G, nx.spring_layout(G, seed=42, k=0.5), node_size=400, edge_color='gray', arrows=True, arrowsize=10, alpha=0.7)
plt.title('Website Hyperlink Structure'); plt.axis('off'); plt.tight_layout(); plt.show()